# 01 - Data Exploration

Characterises the three **romanised Hinglish** corpora this project uses, so the design
decisions in `02_baseline` and later notebooks are motivated by evidence rather than assumed.

Datasets (per the proposal, Technologies -> Datasets):

| Dataset | Role | Expected balance |
|---|---|---|
| **Bohra 2018** | primary, standalone tweets | ~4,575 rows, 36% hate |
| **HASOC 2021 ICHCL** | training + cross-dataset test | ~48% hate |
| **HASOC 2022 ICHCL** | training + cross-dataset test | ~51% hate |

What each section feeds into:
- class balance -> **macro-F1** headline + `class_weight='balanced'` (Objective 2)
- script mix -> the **romanised filter** (this project targets Latin-script Hinglish)
- base-rate + vocabulary divergence across corpora -> **cross-dataset generalisation** (Objective 3)
- romanisation variation in the samples -> the **romanisation-variant attack** (Objective 4)

It uses the same `hinglish_hate` package and `DATA_ROOT` as `02_baseline`, so the two notebooks
describe exactly the same data.

> The Hugging Face `manueltonneau/india-hate-speech-superset` explored in the *first* version of
> this notebook is **not** used here; the appendix documents why it was rejected.

### 1. Setup

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Not on Colab / already mounted:', e)

Not on Colab / already mounted: No module named 'google.colab'


In [2]:
import sys, os
sys.path.insert(0, '/content/drive/MyDrive/dissertation/notebooks')

from hinglish_hate import (
    load_bohra, load_hasoc2021, load_hasoc2022,
    build_corpus, filter_romanised, script_profile, clean_text,
)
from hinglish_hate.loaders import _walk_thread, _to_binary, _finish
import pandas as pd, numpy as np, json
from pathlib import Path
import matplotlib.pyplot as plt
print('imports OK')

imports OK


### 2. Locate data (same convention as `02_baseline`)

In [3]:
DATA_ROOT = Path('/content/drive/MyDrive/dissertation/data')

def find_one(root, name):
    hits = list(Path(root).rglob(name))
    if not hits:
        raise FileNotFoundError(f'{name} not found under {root}')
    return hits[0]

bohra_path = find_one(DATA_ROOT, 'hate_speech.tsv')
h21_labels = list(Path(DATA_ROOT).rglob('labels.json'))
h21_root   = h21_labels[0].parents[2] if h21_labels else None
print('Bohra   :', bohra_path)
print('HASOC21 :', h21_root)

FileNotFoundError: hate_speech.tsv not found under \content\drive\MyDrive\dissertation\data

Clean per-tweet HASOC 2022 loader (same one `02_baseline` uses - single utterances, not
accumulated thread context). Kept inline here so `01` matches `02`; fold into `loaders.py` when
convenient.

In [ ]:
def load_hasoc2022_threads(root):
    root = Path(root)
    thread_dirs = {p.parent for p in root.rglob('binary_labels.json')
                   if (p.parent / 'data.json').exists()}
    rows = []
    for d in sorted(thread_dirs):
        data   = json.load(open(d / 'data.json', encoding='utf-8'))
        labels = json.load(open(d / 'binary_labels.json', encoding='utf-8'))
        id2text = {}
        _walk_thread(data, id2text)
        for tid, lab in labels.items():
            t = id2text.get(str(tid))
            if t:
                rows.append((t, _to_binary(lab)))
    return _finish(rows, 'hasoc2022')

### 3. Load the three corpora

In [ ]:
bohra = load_bohra(bohra_path)
h21   = load_hasoc2021(h21_root) if h21_root else None
h22   = load_hasoc2022_threads(DATA_ROOT)

sources = {'bohra2018': bohra, 'hasoc2021': h21, 'hasoc2022': h22}
sources = {k: v for k, v in sources.items() if v is not None}
for k, v in sources.items():
    print(f'{k:11s} loaded: {len(v):5d} rows')

## 4. Size and class balance

Motivates the headline metric. The hate rate differs by source (Bohra ~0.36 vs HASOC ~0.5), and
each source is internally imbalanced, so **accuracy would flatter a majority-class guesser**.
That is why `02` reports **macro-F1** and trains with `class_weight='balanced'`.

In [ ]:
overview = pd.DataFrame({
    'rows':   {k: len(v) for k, v in sources.items()},
    'hate':   {k: int(v['label'].sum()) for k, v in sources.items()},
    'not':    {k: int((v['label']==0).sum()) for k, v in sources.items()},
    'hate_%': {k: round(v['label'].mean(), 3) for k, v in sources.items()},
})
display(overview)

ax = overview[['not','hate']].plot(kind='bar', stacked=True, figsize=(7,4),
                                   color=['#4c72b0','#c44e52'], edgecolor='black')
ax.set_ylabel('posts'); ax.set_title('Class balance per corpus'); plt.tight_layout(); plt.show()

## 5. Script mix -> the romanised filter

This project targets **romanised** (Latin-script) Hinglish. `script_profile` classifies each post
as `mostly_latin`, `mixed_script`, or `mostly_devanagari`. Bohra is essentially all romanised;
the HASOC corpora carry a Devanagari tail that the romanised filter removes. This is the evidence
behind `filter_romanised`.

In [ ]:
script_tab = pd.DataFrame({k: v['script'].value_counts() for k, v in sources.items()}).fillna(0).astype(int)
display(script_tab)

romanised = {k: len(filter_romanised(v, include_mixed=True)) for k, v in sources.items()}
cov = pd.DataFrame({'total': {k: len(v) for k,v in sources.items()},
                    'romanised_kept': romanised})
cov['kept_%'] = (cov['romanised_kept']/cov['total']).round(3)
print('Coverage after romanised filter (include_mixed=True):')
display(cov)

## 6. Post length

Length distribution per corpus and class. Useful sanity check on the loaders (no empty or
runaway rows) and context for later token-budget / truncation choices in the transformer models.

In [ ]:
def add_len(df):
    d = df.copy()
    d['n_words'] = d['text'].str.split().str.len()
    d['n_chars'] = d['text'].str.len()
    return d

alld = pd.concat([add_len(v) for v in sources.values()], ignore_index=True)
print(alld.groupby('source')[['n_words','n_chars']].describe().round(1)[['n_words']])

fig, ax = plt.subplots(figsize=(7,4))
for k, v in sources.items():
    add_len(v)['n_words'].clip(upper=60).hist(bins=30, alpha=0.5, label=k, ax=ax)
ax.set_xlabel('words per post'); ax.set_ylabel('count'); ax.legend(); ax.set_title('Post length')
plt.tight_layout(); plt.show()

## 7. Sample posts

Qualitative feel for the data: code-switching, and the **romanisation variation** (kya/kyaa/kia)
that Objective 4's novel attack is built on.

In [ ]:
for k, v in sources.items():
    print(f'\n===== {k} =====')
    for lab in (1, 0):
        ex = v[v['label']==lab]['text'].head(3).tolist()
        print(f'  label={lab} ({"hate" if lab else "not"}):')
        for t in ex:
            print('   -', t[:140])

## 8. Cross-dataset divergence -> why generalisation is hard (Objective 3)

Two corpora can share a task yet differ in base rate and vocabulary. Below: the hate base-rate
gap, and the pairwise **Jaccard overlap of the top-500 tokens** on the romanised text. Low overlap
and shifting base rates are exactly why a model trained on one corpus is expected to drop on
another - the effect `02`'s cross-dataset cells measure.

In [ ]:
from collections import Counter

def top_tokens(df, n=500):
    c = Counter()
    for t in filter_romanised(df, include_mixed=True)['text']:
        c.update(clean_text(t).split())
    return set([w for w, _ in c.most_common(n)])

tops = {k: top_tokens(v) for k, v in sources.items()}
keys = list(tops)
jac = pd.DataFrame(index=keys, columns=keys, dtype=float)
for a in keys:
    for b in keys:
        inter = len(tops[a] & tops[b]); union = len(tops[a] | tops[b])
        jac.loc[a, b] = round(inter/union, 3) if union else 0.0
print('Base rate (hate %):', {k: round(v["label"].mean(),3) for k,v in sources.items()})
print('\nTop-500 token Jaccard overlap:')
display(jac)

## 9. Hinglish hate lexicon

The statistical baseline (Objective 2) is *TF-IDF **and lexicon*** + logistic regression. Quick
look at the lexicon that feeds it.

In [ ]:
lex_path = list(DATA_ROOT.rglob('hate_lexicon.txt'))
if lex_path:
    terms = [w.strip() for w in open(lex_path[0], encoding='utf-8') if w.strip()]
    print(f'{len(terms)} lexicon terms. Sample:', terms[:15])
else:
    print('hate_lexicon.txt not found under DATA_ROOT')

## 10. Summary -> design decisions

- Three romanised corpora, all internally imbalanced and with differing hate base rates
  -> **macro-F1** headline, `class_weight='balanced'`.
- HASOC corpora contain a Devanagari tail -> **romanised filter** before every experiment.
- Low cross-corpus token overlap + base-rate shift -> a real **generalisation gap** to measure
  (Objective 3), not an artefact.
- Visible romanisation variation in samples -> basis for the **romanisation-variant attack**
  (Objective 4).

These frames flow directly into `02_baseline` (same package, same `DATA_ROOT`).

## Appendix - why the Hugging Face superset was excluded

The first draft of this notebook explored `manueltonneau/india-hate-speech-superset`. Script
profiling showed it is overwhelmingly **Devanagari**, not romanised Hinglish, so it does not match
the variety this project targets and is excluded. Reproduced here as an audit trail (needs
`datasets`; may be slow).

In [ ]:
# Optional audit - uncomment to reproduce the rejection finding.
# from datasets import load_dataset
# ds = load_dataset('manueltonneau/india-hate-speech-superset')
# sup = pd.DataFrame(ds['train'])
# sup['script'] = sup['text'].map(script_profile)
# print(sup['script'].value_counts(normalize=True).round(3))
# -> mostly_devanagari dominates (~0.9), which is why this source is not used.